# EcoSort AI - Object-Level Local Run

## 1. Read the active model and acceptance evidence

The current application uses a frozen, locally executed SigLIP2 object recognizer. It does not retrain on the sixty acceptance photos. Historical trained benchmark classifiers remain separate and their validation/test scores must not be attributed to this model.

In [ ]:
from pathlib import Path
import json
PROJECT = Path.cwd()
assert (PROJECT / 'ecosort_runtime.py').exists(), 'Open this notebook from EcoSort-Local.'
active = json.loads((PROJECT / 'active_model.json').read_text())
print(active)
RESULTS = PROJECT / active['results']
record = 'deployment_approved.json' if active['results'] == 'trained_objects' else 'training_completed.json'
completion = json.loads((RESULTS / record).read_text())
completion

## 2. Load the verified local model

Load pinned model weights and object descriptions. The strongest object match determines the material category, then a separate official-rule module gives conditional disposal guidance. Relative scores are not calibrated probabilities. Ambiguous classifications require manual review.

In [ ]:
import torch
from ecosort_runtime import EcoSort
print('Apple GPU available:', torch.backends.mps.is_available())
engine = EcoSort()
print('Visual classes:', engine.classes)

## 3. Open the English application

Reuse the running local application, or start it if absent. Classification remains separate from user-confirmed item guidance. Official City of Sydney and NSW EPA links accompany conditional disposal routes.

In [ ]:
from ecosort_runtime import application_running, launch
import webbrowser
if not application_running():
    demo = launch(prevent_thread_lock=True)
webbrowser.open('http://127.0.0.1:7861')

## 4. Inspect all sixty acceptance outcomes

Three fixed panels contain twenty whole web photos each. Each panel needs at least seventeen correct category-and-route outputs. Uncertain outputs count as failures. This is a stratified convenience sample, not a guarantee for arbitrary phone photos. The six local development photos are not independent acceptance evidence.

In [ ]:
import pandas as pd
report = json.loads((PROJECT / 'acceptance_web/results.json').read_text())
display(pd.DataFrame(report['groups']))
display(pd.read_csv(PROJECT / 'acceptance_web/results.csv'))
print(report['warning'])